# ランキング TOP10 画像 一括生成

TSV を読み込み、**playlist_name のパターンを全部抽出して**プレイリストごとに
再生数の伸び TOP10 の画像を作り、ZIP でまとめてダウンロードする。
デザインモチーフは**五線譜**（白背景＋横黒線）。

仕様書: [ranking_top10_spec.md](ranking_top10_spec.md)

## 使い方

1. TSV を用意する（Colab の場合は左のファイルペインから `/content/` にアップロード）
2. **セル「① 設定」だけ**を編集する
3. 上から順に全セルを実行する
4. ⑥ の実行が終わると `ranking_images.zip` のダウンロードが始まる

## 生成されるもの

1 プレイリストにつき **回答版 1 枚 + クイズ版 3 枚**（既定は 2位 / 4位 / 9位）。

| ファイル名 | 内容 |
|---|---|
| `ranking_top10_{プレイリスト}_answer.png` | 回答発表版（何も隠さない） |
| `ranking_top10_{プレイリスト}_quiz2.png` | 2 位を隠したクイズ版 |
| `ranking_top10_{プレイリスト}_quiz4.png` | 4 位を隠したクイズ版 |
| `ranking_top10_{プレイリスト}_quiz9.png` | 9 位を隠したクイズ版 |
| `answer_key.txt` | 隠した動画のタイトル一覧（答え合わせ用） |

隠す順位は `MASK_RANKS = [2, 4, 9]` で変えられる。
母集団が足りず存在しない順位は自動でスキップされる。

**カードの高さは隠す／隠さないに関係なく同じ計算**なので、回答版とクイズ版は
レイアウトが 1px もずれない（クイズ→回答の切り替え投稿で画像が同じ位置で入れ替わる）。

## 対象プレイリストの絞り込み

| 設定 | 意味 |
|---|---|
| `ONLY_PLAYLISTS` | `None` なら全パターン。`["シクフォニの日常"]` のように書くとそれだけ生成（試作用） |
| `MIN_POOL` | 母集団がこの本数未満のプレイリストはスキップ（既定 10） |
| `INCLUDE_EMPTY_PLAYLIST` | `playlist_name` が空欄の行を対象にするか（既定 `False`） |

③ のセルが、どのプレイリストが対象／スキップかを一覧で表示する。

## 入力 TSV の形式

タブ区切り・ヘッダー行あり・UTF-8。以下の列を使用する。

| 列 | 用途 |
|---|---|
| `thumbnail_url` | サムネイル取得の第 1 候補 URL |
| `videoid` | フォールバック URL の組み立て／重複検知 |
| `title` | カードのタイトル |
| `publish_date` | タイトル末尾に `（2023.06.04）` として連結 |
| `from_view_date` / `to_view_date` | サブタイトルの自動生成 |
| `view_diff` | ランキングのソートキー（降順） |
| `playlist_name` | プレイリストの識別 |

`from_view_count` / `to_view_count` は読み込むが使用しない。

> **プレイリスト単位で処理するのは必須。** プレイリストを横断すると同一動画が
> 複数行（所属プレイリスト違い）で現れ、同じ動画が重複してランクインしてしまう。

## 注意

- サムネイル取得に**インターネット接続が必要**。取得できなかった動画はグレーの
  ダミー画像になり、警告が表示される。1 度取得したサムネはキャッシュされる。
- Shorts のサムネイルは 16:9 の中に縦動画が収まった形（左右が黒帯）になる。
  これは YouTube 側の仕様で、本ツールでは黒帯を除去していない。
- **TSV を Excel で開いたら保存せずに閉じること。** `publish_date` が
  `2023-05-12` → `5/12/2023` に化ける。コード側で吸収しているが警告が出る。

## ⓪ 環境準備

In [ ]:
# Colab では毎回必要。ローカルで導入済みならこの行はコメントアウトしてよい
!pip install -q japanize-matplotlib

import csv
import datetime
import os
import re
import unicodedata
import zipfile
from io import BytesIO

import matplotlib.patches as patches
import matplotlib.patheffects as path_effects
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from PIL import Image

import japanize_matplotlib  # noqa: F401  import するだけで日本語フォントが有効になる

## ① 設定 ★ 運用時に編集するのはこのセルだけ

In [ ]:
# 入力 TSV のパス（Colab なら "/content/202608_sixfonia_ranking_quiz.tsv" など）
TSV_PATH = "202608_sixfonia_ranking_quiz.tsv"

# 上位何件を出すか
TOP_N = 10

# ==============================================================
# 一括生成の対象
# ==============================================================
# TSV の playlist_name を全パターン抽出して順に画像化する。
# 特定のプレイリストだけ試したいときは ["シクフォニの日常"] のようにリストで指定。
# None なら全パターンが対象。
ONLY_PLAYLISTS = None

# 母集団がこの本数未満のプレイリストはスキップする。
# TOP10 が成立しない・9位のマスクが作れないため。
MIN_POOL = 10

# playlist_name が空欄の行（どのプレイリストにも属さない動画）を対象にするか
INCLUDE_EMPTY_PLAYLIST = False

# ==============================================================
# クイズ用マスク
# ==============================================================
# 1 プレイリストにつき、ここに挙げた順位ぶんのクイズ画像を作る。
# 母集団が足りず存在しない順位はスキップされる（警告あり）。
MASK_RANKS = [2, 4, 9]

# マスクなしの回答発表版も作るか
MAKE_ANSWER = True

# 隠し方: "mosaic"（モザイク）/ "black"（黒塗り）
MASK_STYLE = "mosaic"

# 何を隠すか: "both"（サムネ+タイトル）/ "thumbnail"（サムネのみ）/ "title"（タイトルのみ）
MASK_TARGET = "both"

# 隠した場所に重ねる文字。空文字 "" にすると何も描かない
MASK_MARK = "？"

# モザイクの粗さ（横方向の分割数。小さいほど粗い）
MOSAIC_BLOCKS = 14

# ==============================================================
# ヘッダー文言
# ==============================================================
# タイトル・サブタイトルとも、以下のプレースホルダが使える:
#   {playlist} … プレイリスト名
#   {pub_from} / {pub_to} … 対象動画の投稿年月（例 2022.08 / 2023.08）
#   {from_date} / {to_date} … 再生数の集計期間（例 2025/08/12 / 2026/08/08）
#   {pool} … 絞り込み後の母集団の本数
#   {n}     … 実際に載る件数
#
# 改行 \n を入れると複数行にできる。長すぎる行は自動で文字サイズを縮める
# （FIG_W=11.5 なら 1 行あたり全角 33.7 文字までが 22pt のまま入る）。
# ※ 下のタイトルは全角34.1字なので、自動縮小で 21.7pt になる（見た目の差はない）
HEADER_TITLE_FMT = "【祝4周年・原点回帰】デビュー年の動画、4年目にたくさん見られたのは？"

# プレイリスト名はここで見せる。非公式の明示も兼ねる。
# ※ 画像はスクショ・転載で単体で流れるので、非公式表記は画像内に残しておく
HEADER_SUB_FMT = "～{playlist}編～(非公式)"

# 画像内のクイズ行。"" にすると行ごと廃止し、高さも確保しない。
# 今回は「設問はツイート本文に書く」方針なので出さない
# （画像の文字とサムネを大きく取るため）
QUIZ_NOTE_FMT = ""

# ==============================================================
# 出力
# ==============================================================
OUT_DIR  = "ranking_images"       # 画像の保存先フォルダ
ZIP_PATH = "ranking_images.zip"   # まとめる ZIP のパス
OUT_DPI  = 260                    # 上げるほど高精細（ファイルサイズも増える）

# 動画タイトル末尾に付ける投稿日の表記
DATE_FORMAT = "（{y}.{m}.{d}）"

## ①-2 ツイート下書きの設定

`answer_key.txt` の末尾に、**データを埋め込んだ状態のツイート本文**を出力する。
当日はここからコピペするだけで済むようにするのが目的。

構成は2段階：

1. **`QUESTIONS`** — 全4問の定義（どのプレイリストの何位を隠すか、本文での表示名）
2. **`TWEET_PLAN`** — 投稿単位。1ツイートに複数問をまとめられる

### 投稿スケジュール

| 日時 | 内容 | 方式 |
|---|---|---|
| 8/12 04:00 | Q1・Q2 を出題 | 予約（単独ツイート） |
| 8/12 09:42 | Q3・Q4 を出題 | 予約（単独ツイート） |
| 8/24 09:42 | 全4問の答え合わせ | 手動（スレッド） |

予約投稿ではスレッドが組めず、先の投稿のURLも貼れないので、
**出題ツイートは時刻で相互に予告する**形にしている。

### 本文で使えるプレースホルダ

出題ツイート（`TWEET_PLAN` の `body`）:

| | 中身 |
|---|---|
| `{questions}` | `QUIZ_LINE_FMT` で組んだ設問行（複数問ぶん） |
| `{answer_at}` | 回答日時 |
| `{post_at}` | その投稿の日時 |

設問行（`QUIZ_LINE_FMT`）と回答本文（`ANSWER_BODY_FMT`）:

| | 中身 | 例 |
|---|---|---|
| `{q}` / `{rank}` | 問番号・隠した順位 | `1` / `9` |
| `{label}` | 本文での表示名（省略時は `playlist`） | `歌ってみた Shorts` |
| `{playlist}` / `{pool}` | TSV上の名前・母集団の本数 | `シクフォニ歌ってみた Shorts` / `16` |
| `{title}` | 正解の動画タイトル | `【期待の新人】一人六役で…` |
| `{pub}` | 投稿日 | `2023.03.28` |
| `{views}` | 集計期間の再生増加数 | `約6.9万回` |
| `{age}` | 投稿からの経過年数 | `3年以上前` |

In [ ]:
# 回答を投稿する日時（出題ツイートの文中に入る）
ANSWER_AT = "8/24 9:42"

# ==============================================================
# 全4問の定義
# ==============================================================
# playlist … TSV の playlist_name と完全一致させること
# rank     … 隠す順位
# label    … ツイート本文での表示名（省略すると playlist をそのまま使う）
QUESTIONS = [
    {"q": 1, "playlist": "シクフォニ歌ってみた Shorts", "rank": 9, "label": "歌ってみた Shorts"},
    {"q": 2, "playlist": "シクフォニの日常",            "rank": 4},
    {"q": 3, "playlist": "シクフォニオリジナル曲",      "rank": 2},  # 仮
    {"q": 4, "playlist": "シクフォニ歌チャレンジ",      "rank": 4},  # 仮
]

# 設問1行の書式
QUIZ_LINE_FMT = "　Q{q}「{label}」{rank}位は？"

# ==============================================================
# 出題ツイート（予約投稿・スレッドは組めないので各本が自己完結する）
# ==============================================================
TWEET_PLAN = [
    {
        "name": "出題 前半（Q1・Q2）",
        "post_at": "8/12 4:00",
        "questions": [1, 2],
        "body": """4周年おめでとうございます🎉

周年のテーマ「原点回帰」にちなみ、
デビュー年の動画の「4年目の再生数」を全4問のランキングクイズにしてみました！

{questions}

リプで予想お待ちしています！
Q3・Q4は本日9:42、答え合わせは{answer_at}に投稿します⏰

※ファンによる非公式の集計です。
※制作の過程で生成AIを利用しています。詳細は下記noteをご確認ください。
（noteのURL）""",
    },
    {
        "name": "出題 後半（Q3・Q4）",
        "post_at": "8/12 9:42",
        "questions": [3, 4],
        "body": """【原点回帰クイズ 後半】

デビュー年の動画の「4年目の再生数」ランキング、残り2問です。

{questions}

リプで予想お待ちしています！
Q1・Q2は本日4:00の投稿にあります。
答え合わせは{answer_at}に全4問まとめて発表します⏰

※ファンによる非公式の集計です。
※制作の過程で生成AIを利用しています。詳細は下記noteをご確認ください。
（noteのURL）""",
    },
]

# ==============================================================
# 回答スレッド（8/24・手動投稿なのでスレッドと引用RTが使える）
# ==============================================================
# 親ツイート（8/12 4:00 の投稿を引用RTして投稿する）
ANSWER_HEAD_TWEET = """【正解発表】

8/12に出した #原点回帰 クイズ、全4問の答え合わせです🎉
予想を送ってくださった皆さんありがとうございました🙏

1問ずつぶら下げていきます👇"""

# 各問の回答（親ツイートへの返信）
ANSWER_BODY_FMT = """【第{q}問】「{label}」

{rank}位の正解は
「{title}」
（{pub}投稿）でした！

{age}の動画が、4年目だけで{views}再生されています📈"""

# 締め（画像なし）
ANSWER_TAIL_TWEET = """全4問、おつきあいありがとうございました🙏

3〜4年前の動画が今も再生され続けていること自体が、いちばんのお祝いだなと思います。
原点があるから今がある、というのを数字で見られて楽しかったです。

改めて4周年おめでとうございます🎉"""

# 8/12 の各出題ツイートに後から手で付ける導線用の返信
# （予約投稿では先の投稿のURLを貼れないため、当日に手動で足す）
LINK_REPLY_LATER = """後半のQ3・Q4を投稿しました👇
（9:42の投稿URL）"""

LINK_REPLY_ANSWER = """答えはこちらで発表しました👇
（8/24 親ツイートのURL）"""

## ② レイアウト定数・配色

長さの単位はすべて**インチ**。図の高さは中身から自動算出する。

In [ ]:
# === 配色（モノクロ）===
BG_COLOR       = "#ffffff"  # 背景
CARD_COLOR     = "#ffffff"  # カード塗り
LINE_COLOR     = "#000000"  # カード枠・五線譜・区切り線
TEXT_COLOR     = "#111111"  # タイトル・1〜3位の順位数字
RANK_SUB_COLOR = "#333333"  # 4位以降の順位数字
SUB_TEXT_COLOR = "#5a5a5a"  # サブタイトル

# === レイアウト（単位はインチ）===
FIG_W          = 9.0   # 図の横幅
LEFT_MARGIN    = 0.60  # 左マージン（五線譜が覗く幅）
RIGHT_MARGIN   = 0.60  # 右マージン（同上）
TOP_MARGIN     = 0.30
BOT_MARGIN     = 0.40
CARD_H         = 1.04  # カード基本高さ（行数が多いと自動で伸びる）
ROW_GAP        = 0.18  # カード間の余白
RANK_W         = 0.92  # 順位数字の領域幅（カード左端から）
DIV_INSET      = 0.16  # 区切り縦線をカード上下から内側に詰める量
THUMB_PAD      = 0.14  # カード上下からサムネまでの余白
THUMB_INSET    = 0.16  # 区切り縦線からサムネ左端までの余白
TITLE_GAP      = 0.16  # サムネ右端からタイトル文字までの余白
CARD_RIGHT_PAD = 0.22  # カード右端からタイトル折り返し位置までの余白
TEXT_VPAD      = 0.12  # カード上下からタイトル文字までの最低余白

# === ヘッダー領域（行数に応じて高さを自動算出する）===
HEADER_TOP_PAD    = 0.12  # 上マージンからタイトル1行目まで
HEADER_TITLE_LH   = 1.30  # タイトルの行間倍率
HEADER_GAP        = 0.16  # タイトルとサブタイトルの間
HEADER_SUB_LH     = 1.35  # サブタイトルの行間倍率
QUIZ_NOTE_GAP     = 0.18  # サブタイトルとクイズ行の間
HEADER_RULE_GAP   = 0.24  # クイズ行と五線譜の間
HEADER_BOTTOM_PAD = 0.22  # 五線譜と1枚目のカードの間

# === フォントサイズ（pt）===
HEADER_TITLE_FS     = 22  # 収まらない場合は自動で縮む
HEADER_TITLE_MIN_FS = 15  # 自動縮小の下限
HEADER_SUB_FS       = 13
HEADER_SUB_MIN_FS   = 10
QUIZ_NOTE_FS        = 16  # 画像内のクイズ行
RANK_FS_TOP3        = 26  # 1〜3位
RANK_FS_OTHER       = 22  # 4位以降
TITLE_FS            = 13  # 動画タイトル（下げると1行に入る文字数が増える）
LINE_SPACING        = 1.2 # 動画タイトル折り返し時の行間

# === 五線譜 ===
STAFF_GAP          = 0.045  # 線間（インチ）
STAFF_LW           = 0.8    # 線の太さ
STAFF_ALPHA        = 0.30   # 背景の五線譜の濃さ
STAFF_HEADER_ALPHA = 0.90   # ヘッダー直下の区切りの濃さ

# === カード枠 ===
CARD_LW = 1.6

# === クイズ用マスクの見た目 ===
MASK_BLACK_FILL  = "#000000"  # MASK_STYLE="black" のときの塗り色
MASK_BAR_GRAY    = "#9a9a9a"  # MASK_STYLE="mosaic" のときのタイトル帯の色
                              # （文字はモザイクにできないので帯で覆う）
MASK_MARK_COLOR  = "#ffffff"  # 「？」の文字色
MASK_MARK_STROKE = "#000000"  # 「？」の縁取り色（モザイクの上でも読めるように）
MASK_MARK_FS     = 26         # サムネに重ねる「？」の文字サイズ
MASK_BAR_MARK_FS = 20         # タイトル帯に重ねる「？」の文字サイズ
QUIZ_NOTE_COLOR  = "#111111"  # 画像内クイズ行の文字色

## ③ TSV 読み込み・絞り込み・ランキング生成

In [ ]:
# TSV に必ず必要な列。1つでも欠けたら分かりやすいエラーで止める
REQUIRED_COLUMNS = [
    "thumbnail_url", "videoid", "title", "publish_date",
    "from_view_date", "to_view_date", "view_diff", "playlist_name",
]

# publish_date が "5/12/2023" のようなスラッシュ区切りだったときの解釈順。
# Excel で TSV を開いて保存し直すと 2023-05-12 が "5/12/2023"(en-US) に化けることがある。
# "12/05/2023"（日月年）で出る環境なら "DMY" に変える。
SLASH_DATE_ORDER = "MDY"


def format_ymd(yyyymmdd):
    """20251231 -> 2025/12/31。想定外の値はそのまま返す"""
    s = str(yyyymmdd).strip()
    if len(s) == 8 and s.isdigit():
        return f"{s[0:4]}/{s[4:6]}/{s[6:8]}"
    return s


def parse_publish_date(value):
    """publish_date を (年, 月, 日) の文字列に正規化する。解釈できなければ None。

    受け付ける形式:
      2023-05-12 / 2023/05/12 / 20230512  … 年が先頭なので曖昧さなし
      5/12/2023                            … Excel 経由で化けた形式。SLASH_DATE_ORDER で解釈
    """
    s = str(value).strip()
    if not s:
        return None

    m = re.match(r"^(\d{4})[-/](\d{1,2})[-/](\d{1,2})$", s)
    if m:
        y, mo, d = m.groups()
        return y, mo.zfill(2), d.zfill(2)

    if re.fullmatch(r"\d{8}", s):
        return s[0:4], s[4:6], s[6:8]

    m = re.match(r"^(\d{1,2})[-/](\d{1,2})[-/](\d{4})$", s)
    if m:
        a, b, y = m.groups()
        mo, d = (a, b) if SLASH_DATE_ORDER == "MDY" else (b, a)
        return y, mo.zfill(2), d.zfill(2)

    return None


def format_publish_date(value):
    """2023-06-04 -> （2023.06.04）。解釈できない値なら空文字を返す"""
    parsed = parse_publish_date(value)
    if parsed is None:
        return ""
    y, mo, d = parsed
    return DATE_FORMAT.format(y=y, m=mo, d=d)


def check_publish_dates(rows):
    """publish_date の形式を検査して警告する。
    Excel 経由で形式が崩れると日付が黙って消えるため、必ず気づけるようにする。"""
    values = [str(v).strip() for v in rows["publish_date"]]
    bad = [v for v in values if parse_publish_date(v) is None]
    nonstd = [v for v in values
              if parse_publish_date(v) is not None and not re.fullmatch(r"\d{4}-\d{2}-\d{2}", v)]

    if bad:
        print(f"[警告] publish_date を解釈できない行が {len(bad)} 件あります（例: {bad[:3]}）。")
        print("        該当行は投稿日なしで描画されます。")
    if nonstd:
        print(f"[警告] publish_date が YYYY-MM-DD 形式でない行が {len(nonstd)} 件あります（例: {nonstd[:3]}）。")
        print(f"        SLASH_DATE_ORDER='{SLASH_DATE_ORDER}' として解釈しました。月日が逆なら 'DMY' に変えてください。")
        print("        ※ Excel で TSV を開いて保存し直すと日付形式が変わります。TSV は元のまま使うのが安全です。")


def load_tsv(tsv_path):
    """TSV を読み込んで列を検証する"""
    # encoding: utf-8-sig にしておくと BOM の有無どちらでも読める
    # quoting : 動画タイトルに " が含まれる（例: 【密着】メンバーに24時間"彼女風LINE"を...）。
    #           TSV はクォートしない形式なので、" を引用符として解釈させない
    df = pd.read_csv(
        tsv_path, sep="\t", dtype=str, encoding="utf-8-sig", quoting=csv.QUOTE_NONE
    ).fillna("")

    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(
            f"TSV に必要な列がありません: {missing}\n"
            f"読み込めた列: {list(df.columns)}\n"
            "※ 列が1つしか読めていない場合はタブ区切りではありません（CSV で保存されていないか確認）。"
        )

    df["view_diff"] = pd.to_numeric(df["view_diff"], errors="coerce").fillna(0).astype(int)
    return df


def select_ranking(df, playlist, top_n):
    """1 プレイリスト分のランキングとヘッダー用メタ情報を返す。

    メタ情報の投稿年月・母集団本数は「上位 N 件」ではなく「絞り込み後の全件」から取る
    （サブタイトルは母集団を説明するものなので）。
    """
    hit = df[df["playlist_name"] == playlist].copy()
    if hit.empty:
        raise ValueError(f"playlist_name='{playlist}' に該当する行がありません。")

    # プレイリストで絞れば本来ユニークになるはず。重複はデータ異常として検知する
    n_dup = int(hit["videoid"].duplicated().sum())
    if n_dup:
        print(f"  [警告] videoid が {n_dup} 件重複しています。各 videoid の先頭行のみ残します。")
        hit = hit.drop_duplicates(subset="videoid", keep="first")

    if hit["from_view_date"].nunique() > 1 or hit["to_view_date"].nunique() > 1:
        print("  [警告] from_view_date / to_view_date が行ごとに異なります。先頭行の値を使います。")

    ym = sorted(f"{p[0]}.{p[1]}" for p in
                (parse_publish_date(v) for v in hit["publish_date"]) if p)
    meta = {
        "playlist": playlist,
        "pool": len(hit),
        "pub_from": ym[0] if ym else "",
        "pub_to": ym[-1] if ym else "",
        "from_date": format_ymd(hit["from_view_date"].iloc[0]),
        "to_date": format_ymd(hit["to_view_date"].iloc[0]),
    }

    # kind="stable": view_diff が同値のとき TSV の並び順を保ち、実行ごとに順位が入れ替わらないようにする
    hit = hit.sort_values("view_diff", ascending=False, kind="stable")
    top = hit.head(top_n).reset_index(drop=True)
    meta["n"] = len(top)
    return top, meta


def build_items(rows):
    """描画用のリストに変換する。

    title     … 投稿日を末尾に連結した描画用の文字列
    raw_title … TSV のままのタイトル。answer_key.txt の動画ID一覧で使う
                （再生データ側と突き合わせるので、日付を足していない元の文字列が必要）
    """
    return [
        {
            "rank": i + 1,
            "video_id": row["videoid"],
            "thumbnail_url": row["thumbnail_url"],
            "raw_title": row["title"],
            "publish_date": row["publish_date"],
            "view_diff": int(row["view_diff"]),
            "title": row["title"] + format_publish_date(row["publish_date"]),
        }
        for i, row in rows.iterrows()
    ]


def format_header(fmt, meta, **extra):
    """ヘッダー文言のプレースホルダを展開する"""
    values = {**meta, **extra}
    try:
        return fmt.format(**values)
    except KeyError as e:
        raise ValueError(
            f"ヘッダー文言に使えないプレースホルダ {e} が含まれています。\n"
            f"使えるのは: {', '.join('{' + k + '}' for k in values)}"
        ) from None


# ============ 読み込みとプレイリストの棚卸し ============
DF = load_tsv(TSV_PATH)
check_publish_dates(DF)

# playlist_name のパターンを TSV の出現順で抽出する
ALL_PLAYLISTS = list(dict.fromkeys(DF["playlist_name"]))
if not INCLUDE_EMPTY_PLAYLIST:
    ALL_PLAYLISTS = [p for p in ALL_PLAYLISTS if p]

TARGET_PLAYLISTS, SKIPPED = [], []
for p in ALL_PLAYLISTS:
    pool = int(DF[DF["playlist_name"] == p]["videoid"].nunique())
    if ONLY_PLAYLISTS is not None and p not in ONLY_PLAYLISTS:
        SKIPPED.append((p, pool, "ONLY_PLAYLISTS の対象外"))
    elif pool < MIN_POOL:
        SKIPPED.append((p, pool, f"母集団が MIN_POOL={MIN_POOL} 未満"))
    else:
        TARGET_PLAYLISTS.append(p)

if not TARGET_PLAYLISTS:
    raise ValueError(
        "対象のプレイリストが 1 つもありません。MIN_POOL / ONLY_PLAYLISTS の設定を見直してください。\n"
        f"TSV 内のプレイリスト: {ALL_PLAYLISTS}"
    )

per_playlist = (1 if MAKE_ANSWER else 0) + len(MASK_RANKS)
print(f"TSV: {TSV_PATH}  （全 {len(DF)} 行 / 動画 {DF['videoid'].nunique()} 本）")
print(f"対象プレイリスト {len(TARGET_PLAYLISTS)} 件 × 最大 {per_playlist} 枚 "
      f"= 最大 {len(TARGET_PLAYLISTS) * per_playlist} 枚を生成します\n")
print(f"{'プレイリスト':<26} {'本数':>5}  判定")
print("-" * 70)
for p in TARGET_PLAYLISTS:
    pool = int(DF[DF['playlist_name'] == p]['videoid'].nunique())
    short = [r for r in MASK_RANKS if r > min(pool, TOP_N)]
    note = "対象" if not short else f"対象（{short} 位は本数不足でスキップ）"
    print(f"{p:<26} {pool:>5}  {note}")
for p, pool, why in SKIPPED:
    label = p if p else "(空欄)"
    print(f"{label:<26} {pool:>5}  スキップ: {why}")

## ④ ユーティリティ（日本語折り返し・サムネイル取得）

In [ ]:
# 対応の取れたブロックを 1 トークンとして扱う括弧。動画タイトルは【】区切りが
# 構造的な意味を持つため、この単位で改行したほうが読みやすい
BRACKET_PAIRS = {"【": "】", "『": "』", "「": "」", "（": "）", "《": "》", "［": "］"}


def _char_units(c):
    """文字の表示幅。全角=1.0、半角≒0.55 で近似。
    ※ 大文字の連続（例: SHALL WE GONG）は実際の幅が 0.55 より広いため、
       この近似だと折り返し位置がやや右に伸びる。CARD_RIGHT_PAD の余裕で吸収する想定。"""
    return 1.0 if unicodedata.east_asian_width(c) in ("W", "F", "A") else 0.55


def _tokenize(text):
    """折り返しの最小単位に分割する。
    - 対応の取れた括弧ブロック（【】『』「」（）《》［］）は 1 トークン
    - #/@ で始まる語は次の空白までを 1 トークン（日本語ハッシュタグも割らない）
    - 英数字・記号(_-.')の連なりも 1 トークン
    - それ以外（日本語など）は 1 文字ずつ
    """
    tokens, i, n = [], 0, len(text)
    wordset = "_-.'"
    while i < n:
        c = text[i]
        if c in BRACKET_PAIRS:
            close = text.find(BRACKET_PAIRS[c], i + 1)
            if close != -1:
                tokens.append(text[i:close + 1])
                i = close + 1
                continue
            tokens.append(c)  # 閉じ括弧が無い場合は通常の 1 文字として扱う
            i += 1
        elif c in "#@":
            j = i + 1
            while j < n and not text[j].isspace():
                j += 1
            tokens.append(text[i:j])
            i = j
        elif c.isascii() and (c.isalnum() or c in wordset):
            j = i
            while j < n and text[j].isascii() and (text[j].isalnum() or text[j] in wordset):
                j += 1
            tokens.append(text[i:j])
            i = j
        else:
            tokens.append(c)
            i += 1
    return tokens


def wrap_jp(text, max_units):
    """日本語テキストを max_units（全角換算の幅）で折り返す。
    括弧ブロック・英単語・ハッシュタグは途中で割らない。"""
    def uw(s):
        return sum(_char_units(c) for c in s)

    out, cur, used = [], "", 0.0
    for tok in _tokenize(text):
        if tok == "\n":
            out.append(cur)
            cur, used = "", 0.0
            continue
        if cur == "" and tok.isspace():  # 行頭の空白は捨てる（全角スペース U+3000 も含む）
            continue
        w = uw(tok)
        if w > max_units and len(tok) > 1:  # 1 行に収まらない長トークンは 1 文字ずつ
            for ch in tok:
                cw = _char_units(ch)
                if cur and used + cw > max_units:
                    out.append(cur)
                    cur, used = "", 0.0
                cur += ch
                used += cw
            continue
        if cur and used + w > max_units:
            out.append(cur)
            if tok.isspace():
                cur, used = "", 0.0
            else:
                cur, used = tok, w
        else:
            cur += tok
            used += w
    if cur:
        out.append(cur)
    return "\n".join(s.rstrip() for s in out) if out else text


def _crop_16x9(img):
    """4:3 など黒帯入りのサムネを中央基準で 16:9 に切り出す（拡大はしない）。
    ※ Shorts の縦動画サムネは 16:9 の中に左右黒帯付きで入っているが、
       それは YouTube 側の仕様なのでここでは除去しない"""
    w, h = img.size
    target = 16 / 9
    if w / h > target:      # 横長すぎ → 左右をカット
        nw = round(h * target)
        x = (w - nw) // 2
        img = img.crop((x, 0, x + nw, h))
    elif w / h < target:    # 縦長（4:3 の黒帯）→ 上下をカット
        nh = round(w / target)
        y = (h - nh) // 2
        img = img.crop((0, y, w, y + nh))
    return img


_THUMB_CACHE = {}  # video_id -> ndarray。セル再実行時の再ダウンロードを避ける


def load_thumb(video_id, thumbnail_url=None):
    """サムネイルを取得。TSV の URL を第 1 候補にし、失敗したら videoid から
    解像度違いを順に試す。取得できた最大解像度をそのまま返す（縮小は描画側）。
    全滅時はダミー画像を返して処理を止めない。"""
    if video_id in _THUMB_CACHE:
        return _THUMB_CACHE[video_id]

    urls = [thumbnail_url] if thumbnail_url else []
    urls += [
        f"https://img.youtube.com/vi/{video_id}/maxresdefault.jpg",  # 1280x720
        f"https://img.youtube.com/vi/{video_id}/sddefault.jpg",      # 640x480
        f"https://img.youtube.com/vi/{video_id}/hqdefault.jpg",      # 480x360
        f"https://img.youtube.com/vi/{video_id}/mqdefault.jpg",      # 320x180
    ]
    for url in dict.fromkeys(urls):  # 重複 URL を除きつつ順序は維持
        try:
            res = requests.get(url, timeout=10)
            if res.status_code == 200:
                img = Image.open(BytesIO(res.content)).convert("RGB")
                if img.size[0] >= 120:  # 取得失敗時の小さなプレースホルダを除外
                    arr = np.array(_crop_16x9(img))
                    _THUMB_CACHE[video_id] = arr
                    return arr
        except Exception:
            continue

    print(f"[警告] サムネイルを取得できませんでした: {video_id}")
    return np.full((720, 1280, 3), 210, dtype=np.uint8)  # 失敗はキャッシュしない


def pixelate(arr, blocks):
    """モザイク化。いったん blocks 個まで縮小してから最近傍で元サイズへ戻す。
    元画像の解像度に関係なく、見た目のブロック数が blocks で一定になる。"""
    img = Image.fromarray(arr)
    w, h = img.size
    bw = max(1, int(blocks))
    bh = max(1, round(bw * h / w))
    small = img.resize((bw, bh), Image.BILINEAR)
    return np.array(small.resize((w, h), Image.NEAREST))


def apply_mask_to_thumb(arr, style, blocks):
    """クイズ用にサムネを隠す。style は "mosaic" か "black" """
    if style == "black":
        return np.zeros_like(arr)
    if style == "mosaic":
        return pixelate(arr, blocks)
    raise ValueError(f'MASK_STYLE は "mosaic" か "black" を指定してください（現在: {style!r}）')

## ⑤ 描画部品

図を組み立てる関数の定義だけ。実行は次の ⑥ でまとめて行う。

In [ ]:
def draw_staff(ax, y_center, x0, x1, fig_h, alpha, zorder):
    """五線譜（5 本 1 組）を描く。y_center は figure 比率での中心位置"""
    gap = STAFF_GAP / fig_h
    for k in range(-2, 3):
        y = y_center + k * gap
        ax.plot([x0, x1], [y, y], color=LINE_COLOR, linewidth=STAFF_LW,
                alpha=alpha, zorder=zorder, solid_capstyle="butt")


def fit_fontsize(text, max_inch, base_fs, min_fs):
    """text が max_inch に収まるようフォントサイズを下げて返す。
    幅は _char_units の近似（全角=1.0 / 半角≒0.55）で見積もる。
    複数行の場合はいちばん長い行に合わせる。"""
    widest = max((sum(_char_units(c) for c in line) for line in text.split("\n")), default=0.0)
    if widest <= 0:
        return base_fs
    needed_inch = widest * (base_fs / 72.0)
    if needed_inch <= max_inch:
        return base_fs
    return max(min_fs, base_fs * max_inch / needed_inch)


def draw_mask_mark(artist_ax, x, y, fontsize, transform):
    """マスクの上に重ねる「？」。縁取りを付けてモザイクの上でも読めるようにする"""
    if not MASK_MARK:
        return
    txt = artist_ax.text(x, y, MASK_MARK, fontsize=fontsize, fontweight="bold",
                         color=MASK_MARK_COLOR, ha="center", va="center",
                         transform=transform, zorder=6)
    txt.set_path_effects([
        path_effects.withStroke(linewidth=3, foreground=MASK_MARK_STROKE)
    ])


def validate_mask(items, mask_rank):
    """マスク指定を検査する。問題なければ隠す動画のタイトルを返す"""
    if mask_rank is None:
        return None
    if MASK_STYLE not in ("mosaic", "black"):
        raise ValueError(f'MASK_STYLE は "mosaic" か "black" です（現在: {MASK_STYLE!r}）')
    if MASK_TARGET not in ("both", "thumbnail", "title"):
        raise ValueError(
            f'MASK_TARGET は "both" / "thumbnail" / "title" です（現在: {MASK_TARGET!r}）'
        )
    hit = next((it for it in items if it["rank"] == mask_rank), None)
    if hit is None:
        raise ValueError(
            f"MASK_RANK={mask_rank} は範囲外です。指定できるのは 1〜{len(items)} 位です。"
        )
    return hit["title"]


def build_figure(items, header_title, header_sub, mask_rank=None):
    """1 枚ぶんの Figure を組み立てる。mask_rank=None なら回答発表版"""
    if not items:
        raise ValueError("items が空です。")
    validate_mask(items, mask_rank)

    text_w_inch = FIG_W - LEFT_MARGIN - RIGHT_MARGIN  # ヘッダー文字が使える横幅

    # --- ヘッダー: 収まるフォントサイズを決め、行数から領域の高さを算出する ---
    title_fs = fit_fontsize(header_title, text_w_inch, HEADER_TITLE_FS, HEADER_TITLE_MIN_FS)
    sub_fs = fit_fontsize(header_sub, text_w_inch, HEADER_SUB_FS, HEADER_SUB_MIN_FS)

    title_h = len(header_title.split("\n")) * title_fs * HEADER_TITLE_LH / 72.0
    sub_h = len(header_sub.split("\n")) * sub_fs * HEADER_SUB_LH / 72.0
    staff_h = STAFF_GAP * 4

    # クイズ行のぶんの高さは「回答版でも常に確保する」。
    # そうしないとクイズ版と回答版で図の高さが変わり、切り替え投稿で画像がずれる
    note_text = QUIZ_NOTE_FMT.format(rank=mask_rank) if (QUIZ_NOTE_FMT and mask_rank) else ""
    note_block = (QUIZ_NOTE_GAP + QUIZ_NOTE_FS * 1.35 / 72.0) if QUIZ_NOTE_FMT else 0.0

    header_h = (HEADER_TOP_PAD + title_h + HEADER_GAP + sub_h
                + note_block + HEADER_RULE_GAP + staff_h + HEADER_BOTTOM_PAD)

    # サムネはカード基本高さ基準で固定（カードが縦に伸びても大きさは一定）
    thumb_h = CARD_H - 2 * THUMB_PAD
    thumb_w = thumb_h * 16 / 9  # YouTube サムネは 16:9

    # 動画タイトルの折り返し幅（カード右端の手前まで）
    title_x_inch = LEFT_MARGIN + RANK_W + THUMB_INSET + thumb_w + TITLE_GAP
    avail_inch = (FIG_W - RIGHT_MARGIN - CARD_RIGHT_PAD) - title_x_inch
    max_units = avail_inch / (TITLE_FS / 72.0)  # 1 行に入る全角換算の文字数
    line_h = TITLE_FS * LINE_SPACING / 72.0     # 1 行の高さ（インチ）

    # 各カードのテキストを折り返し、行数からカード高さを決める。
    # マスクの有無に関わらずこの計算は同じなので、クイズ版と回答版でレイアウトが完全に一致する
    entries = []  # (item, wrapped_text, n_lines, card_h)
    for item in items:
        wrapped = wrap_jp(item["title"], max_units)
        n_lines = wrapped.count("\n") + 1
        card_h = max(CARD_H, n_lines * line_h + 2 * TEXT_VPAD)
        entries.append((item, wrapped, n_lines, card_h))

    # --- 全体の高さを算出（カードごとの高さ＋カード間の余白）---
    content_h = sum(e[3] for e in entries) + (len(entries) - 1) * ROW_GAP
    fig_h = TOP_MARGIN + header_h + content_h + BOT_MARGIN

    # インチ→図全体に対する比率へ変換するヘルパー
    def fx(inch):
        return inch / FIG_W

    def fyb(top_inch, h):  # 上端からの距離 → bottom 比率
        return 1 - (top_inch + h) / fig_h

    def fyc(top_inch, h):  # 上端からの距離 → 中心の y 比率
        return 1 - (top_inch + h / 2) / fig_h

    fig = plt.figure(figsize=(FIG_W, fig_h))
    fig.patch.set_facecolor(BG_COLOR)

    # 図全体を覆う 1 枚の描画用 Axes（五線譜・カード・テキストを描く）
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(0, 1)   # 明示指定でオートスケールが切れるので、以降の plot で範囲が動かない
    ax.set_ylim(0, 1)
    ax.axis("off")
    ax.set_facecolor(BG_COLOR)

    card_left = fx(LEFT_MARGIN)
    card_width = fx(FIG_W - LEFT_MARGIN - RIGHT_MARGIN)

    # --- ヘッダー（上から順に積む）---
    d = TOP_MARGIN + HEADER_TOP_PAD
    ax.text(0.5, fyc(d, title_h), header_title,
            fontsize=title_fs, fontweight="bold", color=TEXT_COLOR,
            ha="center", va="center", linespacing=HEADER_TITLE_LH, zorder=5)
    d += title_h + HEADER_GAP

    ax.text(0.5, fyc(d, sub_h), header_sub,
            fontsize=sub_fs, color=SUB_TEXT_COLOR,
            ha="center", va="center", linespacing=HEADER_SUB_LH, zorder=5)
    d += sub_h

    if QUIZ_NOTE_FMT:
        if note_text:  # 回答版では場所だけ空けて何も描かない
            ax.text(0.5, fyc(d + QUIZ_NOTE_GAP, QUIZ_NOTE_FS * 1.35 / 72.0), note_text,
                    fontsize=QUIZ_NOTE_FS, fontweight="bold", color=QUIZ_NOTE_COLOR,
                    ha="center", va="center", zorder=5)
        d += note_block

    d += HEADER_RULE_GAP
    draw_staff(ax, fyc(d, staff_h), card_left, card_left + card_width, fig_h,
               STAFF_HEADER_ALPHA, zorder=3)
    d += staff_h + HEADER_BOTTOM_PAD

    # --- カード本体（上から順に配置）---
    rank_x = fx(LEFT_MARGIN + RANK_W / 2)
    div_x = fx(LEFT_MARGIN + RANK_W)
    thumb_x = fx(LEFT_MARGIN + RANK_W + THUMB_INSET)
    title_x = fx(title_x_inch)
    title_w = fx(avail_inch)

    for item, wrapped, n_lines, card_h in entries:
        y_center = fyc(d, card_h)
        rank = item["rank"]

        masked = (mask_rank is not None and rank == mask_rank)
        mask_thumb = masked and MASK_TARGET in ("both", "thumbnail")
        mask_title = masked and MASK_TARGET in ("both", "title")

        # 背景の五線譜（図の全幅・カードより下のレイヤー）。
        # カードが不透明白で覆うため、実際に線が見えるのは左右マージンの部分だけ
        draw_staff(ax, y_center, 0.0, 1.0, fig_h, STAFF_ALPHA, zorder=1)

        # カード枠
        rect = patches.FancyBboxPatch(
            (card_left, fyb(d, card_h)), card_width, card_h / fig_h,
            boxstyle="round,pad=0.002,rounding_size=0.012",
            facecolor=CARD_COLOR, edgecolor=LINE_COLOR, linewidth=CARD_LW,
            transform=ax.transAxes, mutation_aspect=fig_h / FIG_W, zorder=2,
        )
        ax.add_patch(rect)

        # 順位数字（マスク時も順位は必ず見せる＝これがクイズの設問になる）
        rank_fs = RANK_FS_TOP3 if rank <= 3 else RANK_FS_OTHER
        if len(str(rank)) >= 2:
            rank_fs *= 0.82
        ax.text(rank_x, y_center, str(rank),
                fontsize=rank_fs, fontweight="bold",
                color=TEXT_COLOR if rank <= 3 else RANK_SUB_COLOR,
                ha="center", va="center", zorder=4)

        # 順位とサムネの区切り縦線
        ax.plot([div_x, div_x],
                [1 - (d + card_h - DIV_INSET) / fig_h, 1 - (d + DIV_INSET) / fig_h],
                color=LINE_COLOR, linewidth=0.9, alpha=0.5, zorder=4)

        # サムネイル（カード中央に縦センタリング。専用 Axes に imshow で収める）
        # Axes の縦横比と画像の縦横比がどちらも 16:9 なので、imshow の aspect='equal' で
        # 位置がずれることはない
        thumb_img = load_thumb(item["video_id"], item["thumbnail_url"])
        interp = "lanczos"
        if mask_thumb:
            thumb_img = apply_mask_to_thumb(thumb_img, MASK_STYLE, MOSAIC_BLOCKS)
            # モザイクは最近傍で描かないとブロックが平滑化されて元に戻ってしまう
            interp = "nearest"

        t_top = d + (card_h - thumb_h) / 2
        t_ax = fig.add_axes([thumb_x, fyb(t_top, thumb_h),
                             fx(thumb_w), thumb_h / fig_h], zorder=4)
        t_ax.imshow(thumb_img, interpolation=interp)
        t_ax.axis("off")
        if mask_thumb:
            draw_mask_mark(t_ax, 0.5, 0.5, MASK_MARK_FS, t_ax.transAxes)

        # 動画タイトル
        if mask_title:
            # 文字はモザイクにできないので、折り返し後の行数ぶんの高さの帯で覆う
            bar_h = (n_lines * line_h) / fig_h
            ax.add_patch(patches.Rectangle(
                (title_x, y_center - bar_h / 2), title_w, bar_h,
                facecolor=MASK_BLACK_FILL if MASK_STYLE == "black" else MASK_BAR_GRAY,
                edgecolor="none", transform=ax.transAxes, zorder=4,
            ))
            draw_mask_mark(ax, title_x + title_w / 2, y_center,
                           MASK_BAR_MARK_FS, ax.transAxes)
        else:
            ax.text(title_x, y_center, wrapped, fontsize=TITLE_FS,
                    color=TEXT_COLOR, va="center", ha="left",
                    linespacing=LINE_SPACING, zorder=4)

        d += card_h + ROW_GAP

    return fig


def out_filename(playlist, mask_rank):
    """ファイル名。Windows/macOS で使えない文字は _ に置換する"""
    safe = re.sub(r"[\s/\\:*?\"<>|]+", "_", playlist)
    suffix = f"_quiz{mask_rank}" if mask_rank is not None else "_answer"
    return f"ranking_top10_{safe}{suffix}.png"

## ⑥ 一括生成 → ZIP でダウンロード

全プレイリスト × （回答版 + `MASK_RANKS` のクイズ版）を生成し、`OUT_DIR` に保存して
`ZIP_PATH` にまとめる。Colab なら実行後そのままダウンロードが始まる。

同梱する `answer_key.txt` は3部構成：

| 節 | 内容 | 用途 |
|---|---|---|
| 1. クイズの答え | 全プレイリスト × 全マスク順位の正解 | 手元での確認用 |
| 2. ツイート下書き | `QUIZ_PLAN` にデータを埋めた本文 | **当日そのままコピペ** |
| 3. 動画ID一覧 | `videoid` / `title` のタブ区切り | **スプレッドシートに貼って再生データを絞り込む** |

ツイート下書きには**添付する画像のファイル名**と**X換算の文字数**が付く。
上限（280＝全角140字）を超えた本文には `← 上限超過！要短縮` が付き、実行ログにも警告が出る。

In [ ]:
X_LIMIT = 280  # X の文字数上限（無料枠）。全角は 2 としてカウントされる


def x_length(text):
    """X の文字数カウントの概算。半角=1 / それ以外(日本語・絵文字)=2。
    URL は実際の長さに関係なく一律 23 として数えられる点は考慮していない"""
    return sum(1 if c.isascii() else 2 for c in text)


def format_views(n):
    """68969 -> 約6.9万回"""
    if n >= 10000:
        return f"約{n / 10000:.1f}万回"
    return f"約{n:,}回"


def years_since(publish_date, to_date_str):
    """投稿日から集計終了日までの経過年数 -> '3年以上前'"""
    p = parse_publish_date(publish_date)
    m = re.match(r"(\d{4})/(\d{1,2})/(\d{1,2})", str(to_date_str))
    if not p or not m:
        return ""
    pub = datetime.date(int(p[0]), int(p[1]), int(p[2]))
    end = datetime.date(*(int(g) for g in m.groups()))
    return f"{int((end - pub).days // 365.25)}年以上前"


def resolve_questions(df):
    """QUESTIONS にランキングの実データを埋めて {問番号: 値} を返す。
    解決できない問はスキップし、理由を返り値の第2要素に入れる"""
    resolved, problems = {}, []

    for spec in QUESTIONS:
        playlist, rank = spec["playlist"], spec["rank"]

        if playlist not in TARGET_PLAYLISTS:
            problems.append(
                f"Q{spec['q']}: playlist_name='{playlist}' が生成対象にありません。\n"
                f"        対象: {TARGET_PLAYLISTS}"
            )
            continue
        if rank not in MASK_RANKS:
            problems.append(
                f"Q{spec['q']}: {rank}位のクイズ画像が作られていません"
                f"（MASK_RANKS={MASK_RANKS} に {rank} を足してください）"
            )

        rows, meta = select_ranking(df, playlist, TOP_N)
        items = build_items(rows)
        hit = next((it for it in items if it["rank"] == rank), None)
        if hit is None:
            problems.append(f"Q{spec['q']}: {playlist} に {rank}位がありません（掲載{len(items)}件）")
            continue

        p = parse_publish_date(hit["publish_date"])
        resolved[spec["q"]] = {
            **meta,
            **spec,
            "label": spec.get("label", playlist),
            "title": hit["raw_title"],
            "pub": f"{p[0]}.{p[1]}.{p[2]}" if p else "",
            "views": format_views(hit["view_diff"]),
            "age": years_since(hit["publish_date"], meta["to_date"]),
            "quiz_image": out_filename(playlist, rank),
            "answer_image": out_filename(playlist, None),
        }

    return resolved, problems


def build_tweet_drafts(df):
    """TWEET_PLAN にデータを埋めて、そのまま貼れるツイート本文を組み立てる"""
    out = []
    resolved, problems = resolve_questions(df)

    def block(header, meta_line, body):
        length = x_length(body)
        over = "  ← 280超過（Premium必須）" if length > X_LIMIT else ""
        out.append("-" * 70)
        out.append(header)
        out.append(meta_line)
        out.append(f"文字数: {length}（うちURL分は未計上。実質+23）{over}")
        out.append("-" * 70)
        out.append(body)
        out.append("")

    if problems:
        out.append("【!】設定に問題があります")
        out.extend(f"    {p}" for p in problems)
        out.append("")

    # --- 出題ツイート ---
    out.append("【1】8/12 予約投稿（各本が単独ツイート。スレッドにしない）")
    out.append("")
    for plan in TWEET_PLAN:
        qs = [resolved[q] for q in plan["questions"] if q in resolved]
        lines = "\n".join(QUIZ_LINE_FMT.format(**q) for q in qs)
        body = plan["body"].format(
            questions=lines, answer_at=ANSWER_AT, post_at=plan["post_at"]
        )
        images = "  →  ".join(q["quiz_image"] for q in qs)
        block(
            f"{plan['name']} ｜ {plan['post_at']} 予約",
            f"添付画像（この順番で）: {images}",
            body,
        )

    # --- 回答スレッド ---
    out.append("")
    out.append(f"【2】{ANSWER_AT} 手動投稿（スレッド）")
    out.append("")
    block("親ツイート（8/12 4:00 の投稿を引用RT）", "画像なし", ANSWER_HEAD_TWEET)
    for q in sorted(resolved):
        item = resolved[q]
        block(f"↳ 返信: 第{q}問の回答（{item['label']} / {item['rank']}位）",
              f"添付画像: {item['answer_image']}",
              ANSWER_BODY_FMT.format(**item))
    block("↳ 返信: 締め", "画像なし", ANSWER_TAIL_TWEET)

    # --- 当日に手で付ける返信 ---
    out.append("")
    out.append("【3】当日に手動で付ける返信（予約投稿ではURLを貼れないため）")
    out.append("")
    block("8/12 4:00 の投稿へ（9:42 の投稿後に付ける）", "画像なし", LINK_REPLY_LATER)
    block("8/12 の両方の投稿へ（8/24 の発表後に付ける）", "画像なし", LINK_REPLY_ANSWER)

    return out


def generate_all():
    """全対象プレイリストぶんの画像を生成する。

    戻り値:
      produced     … 生成した画像のパス一覧
      answer_lines … answer_key.txt の答え部分
      id_map       … {videoid: raw_title}。画像に載った動画を出現順に重複排除したもの
    """
    os.makedirs(OUT_DIR, exist_ok=True)
    produced, answer_lines = [], []
    id_map = {}  # dict は挿入順を保つので、プレイリスト順・順位順が維持される

    for idx, playlist in enumerate(TARGET_PLAYLISTS, 1):
        print(f"[{idx}/{len(TARGET_PLAYLISTS)}] {playlist}")
        rows, meta = select_ranking(DF, playlist, TOP_N)
        items = build_items(rows)

        header_title = format_header(HEADER_TITLE_FMT, meta)
        header_sub = format_header(HEADER_SUB_FMT, meta)

        # 画像に載った動画を記録する（同じ動画が複数プレイリストに出たら初出のみ）
        for item in items:
            id_map.setdefault(item["video_id"], item["raw_title"])

        answer_lines.append(f"■ {playlist}（母集団 {meta['pool']}本 / 掲載 {meta['n']}件）")

        # 回答版 → クイズ版の順に作る
        jobs = ([None] if MAKE_ANSWER else []) + list(MASK_RANKS)
        for mask_rank in jobs:
            if mask_rank is not None and mask_rank > len(items):
                print(f"    スキップ: {mask_rank}位は存在しません（掲載 {len(items)} 件）")
                continue

            fig = build_figure(items, header_title, header_sub, mask_rank)
            path = os.path.join(OUT_DIR, out_filename(playlist, mask_rank))
            fig.savefig(path, dpi=OUT_DPI, facecolor=BG_COLOR)
            plt.close(fig)  # 閉じないと図が溜まってメモリを食う

            produced.append(path)
            size_mb = os.path.getsize(path) / 1024 / 1024
            label = "回答版" if mask_rank is None else f"クイズ {mask_rank}位"
            print(f"    {label:<12} {os.path.basename(path)}  ({size_mb:.1f}MB)")

            if mask_rank is not None:
                hidden = next(it for it in items if it["rank"] == mask_rank)
                answer_lines.append(f"    {mask_rank}位: {hidden['video_id']}  {hidden['title']}")

        answer_lines.append("")

    return produced, answer_lines, id_map


def write_answer_key(answer_lines, tweet_lines, id_map, path):
    """答え・ツイート下書き・動画ID一覧を 1 ファイルに書き出す"""
    with open(path, "w", encoding="utf-8") as f:
        f.write(f"原点回帰クイズ 作業用メモ（{TSV_PATH}）\n")

        # --- 1. クイズの答え ---
        f.write("\n" + "=" * 70 + "\n")
        f.write("1. クイズの答え（全プレイリスト・全マスク順位）\n")
        f.write("=" * 70 + "\n")
        f.write("\n".join(answer_lines))

        # --- 2. ツイート下書き ---
        f.write("\n" + "=" * 70 + "\n")
        f.write("2. ツイート下書き（データ埋め込み済み・そのままコピペ可）\n")
        f.write("=" * 70 + "\n")
        f.write("\n".join(tweet_lines))

        # --- 3. 動画ID一覧（タブ区切り）---
        # スプレッドシートに貼ると2列に分かれる。別シートの再生データを
        # videoid で絞り込む（VLOOKUP / FILTER）ための対応表
        f.write("\n" + "=" * 70 + "\n")
        f.write(f"3. 動画ID一覧（スプレッドシート貼り付け用・タブ区切り / {len(id_map)}本）\n")
        f.write("下のヘッダー行から最後までを選択してコピー＆貼り付けしてください\n")
        f.write("title は TSV のままのタイトルです（画像用に付けた投稿日は含みません）\n")
        f.write("=" * 70 + "\n")
        f.write("videoid\ttitle\n")
        for video_id, raw_title in id_map.items():
            f.write(f"{video_id}\t{raw_title}\n")
    return path


def make_zip(paths, answer_key_path, zip_path):
    """生成物を ZIP にまとめる"""
    # PNG は内部で既に圧縮済みなので、再圧縮しても縮まない。無圧縮で速く作る
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_STORED) as zf:
        for p in paths:
            zf.write(p, arcname=os.path.basename(p))
        zf.write(answer_key_path, arcname=os.path.basename(answer_key_path))
    return zip_path


images, answers, id_map = generate_all()

print("\nツイート下書きを作成中...")
tweets = build_tweet_drafts(DF)

answer_key_path = write_answer_key(
    answers, tweets, id_map, os.path.join(OUT_DIR, "answer_key.txt")
)
zip_path = make_zip(images, answer_key_path, ZIP_PATH)

zip_mb = os.path.getsize(zip_path) / 1024 / 1024
print("\n" + "=" * 70)
print(f"画像 {len(images)} 枚 / ツイート下書き / 動画ID {len(id_map)} 本の対応表を出力しました")
print(f"→ {os.path.abspath(zip_path)}  ({zip_mb:.1f}MB)")
if zip_mb > 100:
    print("[注意] ZIP が 100MB を超えています。ダウンロードが不安定なら OUT_DPI を下げてください"
          f"（現在 {OUT_DPI}。200 にすると約 {zip_mb * (200 / OUT_DPI) ** 2:.0f}MB）。")

# 設定の不整合・文字数オーバーはここで気づけるようにする
if any("【!】" in ln for ln in tweets):
    print("\n[警告] ツイート下書きの設定に問題があります。answer_key.txt の冒頭を確認してください。")
over = [ln for ln in tweets if "280超過" in ln]
if over:
    print(f"\n[情報] 280字を超えるツイートが {len(over)} 本あります（X Premium 前提の想定です）。")

# Colab ならそのままダウンロードする
try:
    from google.colab import files
except ImportError:
    print("Colab ではないので自動ダウンロードはしません。上のパスから取得してください。")
else:
    files.download(zip_path)

## ⑦ マスク候補の比較シート

「どのプレイリストで何位を隠すか」を決めるための一覧画像を作る。
**プレイリスト（行）× 順位（列）**のマトリクスで、各マスに
**そこを隠した場合に伏せられる動画**のサムネイル・順位・再生増加数・タイトルを並べる。

今回の割り当ては **4位×2 / 9位×1 / 2位×1**。この画像を見て、
どのプレイリストにどの順位を振ると面白いクイズになるかを決め、
セル ①-2 の `QUIZ_PLAN` に反映する。

出力は `mask_candidates.png`（ZIPには含めず単独で保存・ダウンロード）。

In [ ]:
# ===== 比較シートの設定 =====
# 行に並べるプレイリスト（TSV の playlist_name と完全一致させること）
COMPARE_PLAYLISTS = [
    "シクフォニオリジナル曲",
    "シクフォニ歌チャレンジ",
    "シクフォニ歌ってみた Shorts",
    "シクフォニの日常",
]
# 列に並べる順位
COMPARE_RANKS = [2, 4, 9]

CMP_HEADER_TITLE = "マスク候補の比較"
CMP_HEADER_SUB   = "行=プレイリスト / 列=隠す順位。各マスはそこを伏せた場合に隠れる動画。割り当ては 4位×2・9位×1・2位×1"
CMP_OUT_PATH     = os.path.join(OUT_DIR, "mask_candidates.png")
CMP_DPI          = 200

# ===== レイアウト（インチ）=====
CMP_FIG_W       = 9.0
CMP_MARGIN_X    = 0.50
CMP_TOP_MARGIN  = 0.30
CMP_BOT_MARGIN  = 0.35
CMP_HEADER_H    = 0.95   # タイトル+サブ+五線譜
CMP_COL_GAP     = 0.22   # 列間
CMP_ROW_GAP     = 0.34   # 行間
CMP_LABEL_H     = 0.30   # プレイリスト名の帯
CMP_LABEL_GAP   = 0.10   # 帯とサムネの間
CMP_META_H      = 0.22   # 「4位 ／ +60,893回」の行
CMP_TITLE_LINES = 3      # タイトルの最大行数（超えたら … で切る）

CMP_LABEL_FS = 13
CMP_META_FS  = 8.5
CMP_TITLE_FS = 9
CMP_TITLE_LH = 1.25


def wrap_clip(text, max_units, max_lines):
    """折り返したうえで max_lines 行に収め、あふれたら末尾を … にする"""
    lines = wrap_jp(text, max_units).split("\n")
    if len(lines) > max_lines:
        lines = lines[:max_lines]
        lines[-1] = lines[-1][:-1] + "…"
    return "\n".join(lines)


def build_comparison_figure(df, playlists, ranks):
    n_rows, n_cols = len(playlists), len(ranks)
    if not n_rows or not n_cols:
        raise ValueError("COMPARE_PLAYLISTS / COMPARE_RANKS が空です。")

    # 先にデータを集める（存在しない順位はそのマスだけ空にする）
    grid = []  # [(playlist, meta, {rank: item or None})]
    available = {p for p in df["playlist_name"] if p}
    for playlist in playlists:
        if playlist not in available:
            raise ValueError(
                f"playlist_name='{playlist}' が TSV にありません。\n"
                f"候補: {sorted(available)}"
            )
        rows, meta = select_ranking(df, playlist, TOP_N)
        items = build_items(rows)
        cells = {r: next((it for it in items if it["rank"] == r), None) for r in ranks}
        grid.append((playlist, meta, cells))

    # --- 寸法 ---
    col_w = (CMP_FIG_W - 2 * CMP_MARGIN_X - (n_cols - 1) * CMP_COL_GAP) / n_cols
    thumb_h = col_w * 9 / 16
    title_h = CMP_TITLE_LINES * CMP_TITLE_FS * CMP_TITLE_LH / 72.0
    cell_h = thumb_h + CMP_META_H + title_h
    row_h = CMP_LABEL_H + CMP_LABEL_GAP + cell_h
    fig_h = (CMP_TOP_MARGIN + CMP_HEADER_H + n_rows * row_h
             + (n_rows - 1) * CMP_ROW_GAP + CMP_BOT_MARGIN)

    max_units = col_w / (CMP_TITLE_FS / 72.0)  # 1 行に入る全角換算の文字数

    def fx(inch):
        return inch / CMP_FIG_W

    def fyb(top_inch, h):
        return 1 - (top_inch + h) / fig_h

    def fyc(top_inch, h):
        return 1 - (top_inch + h / 2) / fig_h

    fig = plt.figure(figsize=(CMP_FIG_W, fig_h))
    fig.patch.set_facecolor(BG_COLOR)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
    ax.set_facecolor(BG_COLOR)

    left = fx(CMP_MARGIN_X)
    right = fx(CMP_FIG_W - CMP_MARGIN_X)

    # --- ヘッダー ---
    ax.text(0.5, fyc(CMP_TOP_MARGIN + 0.04, 0.32), CMP_HEADER_TITLE,
            fontsize=20, fontweight="bold", color=TEXT_COLOR,
            ha="center", va="center", zorder=5)
    sub_fs = fit_fontsize(CMP_HEADER_SUB, CMP_FIG_W - 2 * CMP_MARGIN_X, 10, 7)
    ax.text(0.5, fyc(CMP_TOP_MARGIN + 0.40, 0.22), CMP_HEADER_SUB,
            fontsize=sub_fs, color=SUB_TEXT_COLOR, ha="center", va="center", zorder=5)
    draw_staff(ax, fyc(CMP_TOP_MARGIN + CMP_HEADER_H - 0.30, STAFF_GAP * 4),
               left, right, fig_h, STAFF_HEADER_ALPHA, zorder=3)

    # --- 行（プレイリスト）---
    d = CMP_TOP_MARGIN + CMP_HEADER_H
    for playlist, meta, cells in grid:
        # プレイリスト名の帯
        ax.text(left, fyc(d, CMP_LABEL_H), f"{playlist}（母集団 {meta['pool']}本）",
                fontsize=CMP_LABEL_FS, fontweight="bold", color=TEXT_COLOR,
                ha="left", va="center", zorder=5)
        line_y = fyb(d, CMP_LABEL_H)
        ax.plot([left, right], [line_y, line_y], color=LINE_COLOR,
                linewidth=1.0, alpha=0.6, zorder=3)
        d += CMP_LABEL_H + CMP_LABEL_GAP

        for c, rank in enumerate(ranks):
            cell_left = CMP_MARGIN_X + c * (col_w + CMP_COL_GAP)
            item = cells.get(rank)

            if item is None:
                ax.text(fx(cell_left + col_w / 2), fyc(d, thumb_h),
                        f"{rank}位なし", fontsize=CMP_META_FS, color=SUB_TEXT_COLOR,
                        ha="center", va="center", zorder=4)
                continue

            # サムネイル（枠線を残してマスの境目を分かりやすくする）
            t_ax = fig.add_axes([fx(cell_left), fyb(d, thumb_h),
                                 fx(col_w), thumb_h / fig_h], zorder=4)
            t_ax.imshow(load_thumb(item["video_id"], item["thumbnail_url"]),
                        interpolation="lanczos")
            t_ax.set_xticks([])
            t_ax.set_yticks([])
            for spine in t_ax.spines.values():
                spine.set_color(LINE_COLOR)
                spine.set_linewidth(0.8)

            # 「4位 ／ +60,893回」
            ax.text(fx(cell_left), fyc(d + thumb_h, CMP_META_H),
                    f"{rank}位  ／  +{item['view_diff']:,}回",
                    fontsize=CMP_META_FS, fontweight="bold", color=TEXT_COLOR,
                    ha="left", va="center", zorder=4)

            # タイトル（最大 CMP_TITLE_LINES 行）
            ax.text(fx(cell_left), fyb(d + thumb_h + CMP_META_H, 0),
                    wrap_clip(item["raw_title"], max_units, CMP_TITLE_LINES),
                    fontsize=CMP_TITLE_FS, color=TEXT_COLOR,
                    ha="left", va="top", linespacing=CMP_TITLE_LH, zorder=4)

        d += cell_h + CMP_ROW_GAP

    return fig


os.makedirs(OUT_DIR, exist_ok=True)  # ⑥ を飛ばして ⑦ だけ実行した場合に備える
plt.close("all")

cmp_fig = build_comparison_figure(DF, COMPARE_PLAYLISTS, COMPARE_RANKS)
cmp_fig.savefig(CMP_OUT_PATH, dpi=CMP_DPI, facecolor=BG_COLOR)
plt.show()

print("保存しました:", os.path.abspath(CMP_OUT_PATH))
print("\n割り当て: 4位×2 / 9位×1 / 2位×1")
print("決めたら セル ①-2 の QUIZ_PLAN の playlist / rank を書き換えて、⑥ を再実行してください。")

try:
    from google.colab import files
except ImportError:
    pass
else:
    files.download(CMP_OUT_PATH)

## ⑧ 割り当てパターンの一覧

`{4, 9, 4, 2}` を4プレイリストに振り分ける組み合わせを**全パターン1行ずつ**並べる。
4位が2つあるので **4! ÷ 2! = 12通り**。

⑦（プレイリスト×順位のマトリクス）が「各マスに何が入るか」を見るためのものなのに対し、
こちらは**1行＝1つの割り当て案**で、実際に4問セットにしたときの並びを比較する。

気に入ったパターンの番号を控えて、①-2 の `QUIZ_PLAN` の `rank` に反映する。
出力は `mask_patterns.png`。

In [ ]:
from itertools import permutations

# ===== パターン一覧の設定 =====
# 列（左から順）に並べるプレイリスト
PATTERN_PLAYLISTS = COMPARE_PLAYLISTS
# 4 プレイリストに配る順位の内訳。並べ替えの全パターンを出す
PATTERN_RANKS = [4, 9, 4, 2]

PAT_HEADER_TITLE = "割り当てパターン一覧"
PAT_OUT_PATH = os.path.join(OUT_DIR, "mask_patterns.png")
PAT_DPI = 200

# ===== レイアウト（インチ）=====
PAT_FIG_W      = 9.0
PAT_MARGIN_X   = 0.45
PAT_TOP_MARGIN = 0.30
PAT_BOT_MARGIN = 0.35
PAT_HEADER_H   = 0.90   # タイトル+サブ+五線譜
PAT_COLHEAD_H  = 0.40   # プレイリスト名の見出し行
PAT_LABEL_W    = 0.88   # 左端のパターン番号の欄
PAT_COL_GAP    = 0.14
PAT_META_H     = 0.20   # 「4位 +109,642」
PAT_ROW_GAP    = 0.20

PAT_LABEL_FS   = 10
PAT_COLHEAD_FS = 9
PAT_META_FS    = 7.5


def build_pattern_figure(df, playlists, rank_pool):
    n_cols = len(playlists)
    if n_cols != len(rank_pool):
        raise ValueError(
            f"PATTERN_PLAYLISTS({n_cols}件) と PATTERN_RANKS({len(rank_pool)}件) の数を合わせてください。"
        )

    # 重複を除いた並べ替えを、見た目が安定するようソートして列挙
    patterns = sorted(set(permutations(rank_pool)))

    # プレイリストごとに 1 回だけランキングを取る
    pool = {}
    available = {p for p in df["playlist_name"] if p}
    for playlist in playlists:
        if playlist not in available:
            raise ValueError(
                f"playlist_name='{playlist}' が TSV にありません。\n候補: {sorted(available)}"
            )
        rows, meta = select_ranking(df, playlist, TOP_N)
        pool[playlist] = {it["rank"]: it for it in build_items(rows)}

    # --- 寸法 ---
    col_w = (PAT_FIG_W - 2 * PAT_MARGIN_X - PAT_LABEL_W
             - (n_cols - 1) * PAT_COL_GAP) / n_cols
    thumb_h = col_w * 9 / 16
    row_h = thumb_h + PAT_META_H
    n_rows = len(patterns)
    fig_h = (PAT_TOP_MARGIN + PAT_HEADER_H + PAT_COLHEAD_H
             + n_rows * row_h + (n_rows - 1) * PAT_ROW_GAP + PAT_BOT_MARGIN)

    def fx(inch):
        return inch / PAT_FIG_W

    def fyb(top_inch, h):
        return 1 - (top_inch + h) / fig_h

    def fyc(top_inch, h):
        return 1 - (top_inch + h / 2) / fig_h

    def col_left(c):
        return PAT_MARGIN_X + PAT_LABEL_W + c * (col_w + PAT_COL_GAP)

    fig = plt.figure(figsize=(PAT_FIG_W, fig_h))
    fig.patch.set_facecolor(BG_COLOR)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
    ax.set_facecolor(BG_COLOR)

    left = fx(PAT_MARGIN_X)
    right = fx(PAT_FIG_W - PAT_MARGIN_X)

    # --- ヘッダー ---
    ax.text(0.5, fyc(PAT_TOP_MARGIN + 0.04, 0.32), PAT_HEADER_TITLE,
            fontsize=20, fontweight="bold", color=TEXT_COLOR,
            ha="center", va="center", zorder=5)
    sub = (f"{'・'.join(str(r) + '位' for r in sorted(rank_pool))} を "
           f"{n_cols}プレイリストに振り分ける全{n_rows}通り／1行=1案")
    sub_fs = fit_fontsize(sub, PAT_FIG_W - 2 * PAT_MARGIN_X, 10, 7)
    ax.text(0.5, fyc(PAT_TOP_MARGIN + 0.40, 0.22), sub,
            fontsize=sub_fs, color=SUB_TEXT_COLOR, ha="center", va="center", zorder=5)
    draw_staff(ax, fyc(PAT_TOP_MARGIN + PAT_HEADER_H - 0.30, STAFF_GAP * 4),
               left, right, fig_h, STAFF_HEADER_ALPHA, zorder=3)

    # --- 列見出し（プレイリスト名）---
    d = PAT_TOP_MARGIN + PAT_HEADER_H
    for c, playlist in enumerate(playlists):
        head_fs = fit_fontsize(playlist, col_w, PAT_COLHEAD_FS, 5.5)
        ax.text(fx(col_left(c) + col_w / 2), fyc(d, PAT_COLHEAD_H), playlist,
                fontsize=head_fs, fontweight="bold", color=TEXT_COLOR,
                ha="center", va="center", zorder=5)
    d += PAT_COLHEAD_H

    # --- 各パターン ---
    for i, pattern in enumerate(patterns, 1):
        # 左端のパターン番号と順位の並び
        ax.text(fx(PAT_MARGIN_X), fyc(d, thumb_h),
                f"P{i:02d}\n{'/'.join(str(r) for r in pattern)}",
                fontsize=PAT_LABEL_FS, fontweight="bold", color=TEXT_COLOR,
                ha="left", va="center", linespacing=1.4, zorder=5)

        for c, (playlist, rank) in enumerate(zip(playlists, pattern)):
            item = pool[playlist].get(rank)
            cell_x = col_left(c)

            if item is None:
                ax.text(fx(cell_x + col_w / 2), fyc(d, thumb_h), f"{rank}位なし",
                        fontsize=PAT_META_FS, color=SUB_TEXT_COLOR,
                        ha="center", va="center", zorder=4)
                continue

            t_ax = fig.add_axes([fx(cell_x), fyb(d, thumb_h),
                                 fx(col_w), thumb_h / fig_h], zorder=4)
            t_ax.imshow(load_thumb(item["video_id"], item["thumbnail_url"]),
                        interpolation="lanczos")
            t_ax.set_xticks([])
            t_ax.set_yticks([])
            for spine in t_ax.spines.values():
                spine.set_color(LINE_COLOR)
                spine.set_linewidth(0.8)

            ax.text(fx(cell_x), fyc(d + thumb_h, PAT_META_H),
                    f"{rank}位  +{item['view_diff']:,}",
                    fontsize=PAT_META_FS, fontweight="bold", color=TEXT_COLOR,
                    ha="left", va="center", zorder=4)

        d += row_h + PAT_ROW_GAP

    return fig, patterns


os.makedirs(OUT_DIR, exist_ok=True)
plt.close("all")

pat_fig, PATTERNS = build_pattern_figure(DF, PATTERN_PLAYLISTS, PATTERN_RANKS)
pat_fig.savefig(PAT_OUT_PATH, dpi=PAT_DPI, facecolor=BG_COLOR)
plt.show()

print("保存しました:", os.path.abspath(PAT_OUT_PATH))
print(f"\n全 {len(PATTERNS)} パターン（列の順: {' / '.join(PATTERN_PLAYLISTS)}）")
for i, pattern in enumerate(PATTERNS, 1):
    detail = "  ".join(f"{p}={r}位" for p, r in zip(PATTERN_PLAYLISTS, pattern))
    print(f"  P{i:02d}  {'/'.join(str(r) for r in pattern)}   {detail}")
print("\n決めたパターンの順位を ①-2 の QUIZ_PLAN の rank に反映して、⑥ を再実行してください。")

try:
    from google.colab import files
except ImportError:
    pass
else:
    files.download(PAT_OUT_PATH)